In [1]:
import os, sys
import pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath('report.py')))
from report import run

# Number of iterations to average over -- uses only the FIRST K iterations
# of each (type, query, target) group (report.aggregate's n_iterations param).
# The paper reports K=3; change this to see the table at a different K.
K = 5

BENCHMARK = 2

# Which LaTeX table layout to render:
#   1 -> original layout (Type as leftmost column, model names as \multicolumn spanning header rows)
#   2 -> compact layout (Model as leftmost column via \multirow, rotated 90 degrees)
TEMPLATE_VERSION = 2

DATASETS = [("FB", "FB15k-237+H"), ("NELL", "NELL995+H")]
MODELS   = [("cqd", "CQD"), ("betae", "BetaE"), ("query2box", "Query2Box"), ("gqe", "GQE")]

QUERY_ROWS = [
    ("2p",   "2p"),
    ("3p",   "3p"),
    ("2i",   "2i"),
    ("3i",   "3i"),
    ("2u",   "2u"),
    ("2u1p", "up"),
    ("1p2i", "ip"),
    ("2i1p", "pi"),
]

METHODS   = ["First", "Last", "Random", "Shapley"]
COL_ORDER = ["DeltaMRR_Necc", "DeltaMRR_Suff"]


In [2]:
# Collect all data
all_data = {}  # (kg_code, model_code, display_name) -> row dict or None

for kg_code, _ in DATASETS:
    for model_code, _ in MODELS:
        for display_name, csv_suffix in QUERY_ROWS:
            csv_file = f"evaluation_{kg_code}_{BENCHMARK}_{model_code}_{csv_suffix}_random.csv"
            key = (kg_code, model_code, display_name)
            if not os.path.exists(csv_file):
                all_data[key] = None
                continue
            _, _, report_pct = run(csv_file, n_iterations=K)
            row = {}
            for metric in COL_ORDER:
                for m in METHODS:
                    row[(metric, m)] = report_pct.loc[m, metric] if m in report_pct.index else float("nan")
            all_data[key] = row


# Helpers
def fmt(v, bold=False):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return "---"
    s = f"{v:+.2f}"
    return r"\textbf{" + s + "}" if bold else s


def row_cells(model_code, display_name):
    cells = []
    for kg_code, _ in DATASETS:
        key = (kg_code, model_code, display_name)
        row = all_data.get(key)
        for metric in COL_ORDER:
            for m in METHODS:
                v = None if row is None else row.get((metric, m))
                cells.append(fmt(v, bold=(m == "Shapley")))
    return cells


# Render LaTeX
if TEMPLATE_VERSION not in (1, 2):
    raise ValueError(f"Unknown TEMPLATE_VERSION: {TEMPLATE_VERSION!r} (expected 1 or 2)")

lines = []
lines.append(r"\begin{table*}[!t]")
lines.append(r"\centering")
lines.append(
    r"\caption{" + "\n"
    r"Necessity and sufficiency of explanations on the \textbf{FB15k-237+H} and \textbf{NELL995+H} datasets for different" + "\n"
    r"models, query types, and baselines, measured as changes in mean reciprocal rank" + "\n"
    r"($\Delta$MRR, in \%). The most important predicate is substituted with a uniformly" + "\n"
    r"sampled random predicate per iteration; reported values are averaged over all iterations." + "\n"
    r"Lower is better for necessity, higher is better for sufficiency.}"
)
lines.append(r"\label{tab:combined_eval}")
lines.append(r"\footnotesize")
if TEMPLATE_VERSION == 2:
    lines.append(r"\renewcommand{\arraystretch}{0.85}")
lines.append(r"\setlength{\tabcolsep}{3pt}")

if TEMPLATE_VERSION == 1:
    lines.append(r"\begin{tabular}{@{}lrrrrrrrr@{\hspace{20pt}}rrrrrrrr@{}}")
    lines.append(r"\toprule")
    lines.append(
        r"& \multicolumn{8}{c}{\textbf{FB15k-237+H}}"
        r"& \multicolumn{8}{c}{\textbf{NELL995+H}} \\"
    )
    lines.append(r"\cmidrule(lr){2-9} \cmidrule(lr){10-17}")
    lines.append(
        r"\raisebox{-0.8ex}{\textbf{Type}}"
        r"& \multicolumn{4}{c}{\textbf{Necessary}}"
        r"& \multicolumn{4}{c}{\textbf{Sufficient}}"
        r"& \multicolumn{4}{c}{\textbf{Necessary}}"
        r"& \multicolumn{4}{c}{\textbf{Sufficient}} \\"
    )
    lines.append(r"\cmidrule(lr){2-5} \cmidrule(lr){6-9}")
    lines.append(r"\cmidrule(lr){10-13} \cmidrule(lr){14-17}")
    lines.append(
        r"& First & Last & Random & \textbf{CQA-SHAP}"
        r"& First & Last & Random & \textbf{CQA-SHAP}"
        r"& First & Last & Random & \textbf{CQA-SHAP}"
        r"& First & Last & Random & \textbf{CQA-SHAP} \\"
    )
    lines.append(r"\midrule")

    for i, (model_code, model_name) in enumerate(MODELS):
        lines.append(r"\multicolumn{17}{c}{\textbf{" + model_name + r"}} \\")
        lines.append(r"\midrule")
        for display_name, _ in QUERY_ROWS:
            cells = row_cells(model_code, display_name)
            line = r"{\boldmath$" + display_name + r"$}    & " + " & ".join(cells) + r" \\"
            lines.append(line)
        if i < len(MODELS) - 1:
            lines.append(r"\midrule")

elif TEMPLATE_VERSION == 2:
    lines.append(r"\begin{tabular}{@{}l|lrrrrrrrr@{\hspace{6pt}}rrrrrrrr@{}}")
    lines.append(r"\toprule")
    lines.append(
        r"\multicolumn{1}{c}{} & \multicolumn{1}{c}{} & \multicolumn{8}{c}{\textbf{FB15k-237+H}}"
        r"& \multicolumn{8}{c}{\textbf{NELL995+H}} \\"
    )
    lines.append(r"\cmidrule(lr){3-10} \cmidrule(l){11-18}")
    lines.append(
        r"\multicolumn{1}{c}{} & \multicolumn{1}{c}{} & \multicolumn{4}{c}{\textbf{Necessary}}"
        r"& \multicolumn{4}{c}{\textbf{Sufficient}}"
        r"& \multicolumn{4}{c}{\textbf{Necessary}}"
        r"& \multicolumn{4}{c}{\textbf{Sufficient}} \\"
    )
    lines.append(r"\cmidrule(lr){3-6} \cmidrule(lr){7-10}")
    lines.append(r"\cmidrule(lr){11-14} \cmidrule(l){15-18}")
    lines.append(
        r"\multicolumn{1}{c}{} & \textbf{Type}"
        r"& First & Last & Random & \textbf{CQA-SHAP}"
        r"& First & Last & Random & \textbf{CQA-SHAP}"
        r"& First & Last & Random & \textbf{CQA-SHAP}"
        r"& First & Last & Random & \textbf{CQA-SHAP} \\"
    )
    lines.append(r"\midrule")

    for i, (model_code, model_name) in enumerate(MODELS):
        lines.append(r"\multirow{8}{*}{\rotatebox[origin=c]{90}{\textbf{" + model_name + r"}}}")
        for display_name, _ in QUERY_ROWS:
            cells = row_cells(model_code, display_name)
            line = r"    & {\boldmath$" + display_name + r"$}    & " + " & ".join(cells) + r" \\"
            lines.append(line)
        if i < len(MODELS) - 1:
            lines.append(r"\midrule")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"\end{table*}")

print("\n".join(lines))


\begin{table*}[!t]
\centering
\caption{
Necessity and sufficiency of explanations on the \textbf{FB15k-237+H} and \textbf{NELL995+H} datasets for different
models, query types, and baselines, measured as changes in mean reciprocal rank
($\Delta$MRR, in \%). The most important predicate is substituted with a uniformly
sampled random predicate per iteration; reported values are averaged over all iterations.
Lower is better for necessity, higher is better for sufficiency.}
\label{tab:combined_eval}
\footnotesize
\renewcommand{\arraystretch}{0.85}
\setlength{\tabcolsep}{3pt}
\begin{tabular}{@{}l|lrrrrrrrr@{\hspace{6pt}}rrrrrrrr@{}}
\toprule
\multicolumn{1}{c}{} & \multicolumn{1}{c}{} & \multicolumn{8}{c}{\textbf{FB15k-237+H}}& \multicolumn{8}{c}{\textbf{NELL995+H}} \\
\cmidrule(lr){3-10} \cmidrule(l){11-18}
\multicolumn{1}{c}{} & \multicolumn{1}{c}{} & \multicolumn{4}{c}{\textbf{Necessary}}& \multicolumn{4}{c}{\textbf{Sufficient}}& \multicolumn{4}{c}{\textbf{Necessary}}& \multicolumn{4}{c}